kernel: scFates

In [ ]:
%run /work/DevM_analysis/utils/colors.py

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.backends.backend_pdf as mpdf
from matplotlib.pyplot import rc_context

from anndata import AnnData
import scanpy as sc
import scFates as scf
import palantir
import pegasus as pg
import pegasusio as io

import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.filterwarnings(action='once')
warnings.simplefilter(action='once')
warnings.simplefilter(action="ignore", category=NumbaDeprecationWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
#InteractiveShell.ast_node_interactivity = "all" # to show output from all the lines in a cells
pd.set_option('display.max_column', None) # display all the columns in pandas
pd.options.display.max_rows = 100

# Set up

In [ ]:
new_anno = "anno_wnn_v51"
anno_hsc = 'anno_wnn_hsc'
cell = "blood"

In [ ]:
adata = sc.read_h5ad(f'data/FL_{cell}_diffusion.h5ad')
adata

# Palantir

In [ ]:
multivi_projections = pd.DataFrame(adata.obsm["X_multiVI"], index=adata.obs_names)

In [ ]:
dm_res = palantir.utils.run_diffusion_maps(multivi_projections)

In [ ]:
dm_res['EigenValues']

In [ ]:
plt.scatter(np.arange(10), dm_res['EigenValues'])

In [ ]:
ms_data = palantir.utils.determine_multiscale_space(dm_res)

In [ ]:
ms_data.values.shape

# Pegasus

In [ ]:
with rc_context({"figure.figsize": (9, 9)}):
    sc.pl.embedding(adata, basis='fle',
                    color=[f"{new_anno}"],
                    show=False,
                    palette = annot2col)
    plt.savefig(f"plots/FL_blood_mvi_diffusion_FLE.pdf", bbox_inches="tight")

In [ ]:
n_ctypes, ncols = len(adata.obs[new_anno].unique()), 5
nrows = int(np.ceil(n_ctypes / ncols))
sns.set_theme(style="white", font_scale=1.5)
_, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.5, nrows * 5))
for i, clust in enumerate(adata.obs[new_anno].cat.categories):
    i += 1
    sc.pl.embedding(adata, basis='fle', color=[new_anno], groups=[clust],
               show=False, use_raw=False, ax=axes[int((i - 1) / ncols), (i - 1) % ncols],
               title=f"{clust}", palette=['red'], ncols=ncols, legend_loc=None, size=10)
plt.tight_layout()
plt.show()